# 00 · Download the latest official data

Run this once (**Kernel → Restart Kernel and Run All Cells**). Every file is saved into `data/raw/` with the date it was downloaded.
All sources are official statistics agencies; links are kept here so anyone can reproduce the project.

In [1]:
import os, datetime, requests, pandas as pd

RAW = "data/raw"
os.makedirs(RAW, exist_ok=True)
ONS = "https://www.ons.gov.uk/file?uri="
GVA = "/economy/grossdomesticproductgdp/datasets/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiesbyitl1region/"

SOURCES = {
    # --- UK cities ---------------------------------------------------------------
    "uk_population_mid2025.xlsx": ONS + "/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/populationestimatesforukenglandandwalesscotlandandnorthernireland/mid2025/mye25tablesuk.xlsx",
    "uk_gdhi_local_authorities_1997_2024.xlsx": ONS + "/economy/regionalaccounts/grossdisposablehouseholdincome/datasets/regionalgrossdisposablehouseholdincomelocalauthorities/1997to2024/regionalgrossdisposablehouseholdincomelocalauthorities2024.xlsx",
    "uk_ashe_table8_2025_provisional.zip": ONS + "/employmentandlabourmarket/peopleinwork/earningsandworkinghours/datasets/placeofresidencebylocalauthorityashetable8/2025provisional/ashetable82025provisional.zip",
    "uk_gva_london.xlsx":           ONS + GVA + "tlilondon/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestlilondon.xlsx",
    "uk_gva_west_midlands.xlsx":    ONS + GVA + "tlgwestmidlands/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestlgwestmidlands.xlsx",
    "uk_gva_north_west.xlsx":       ONS + GVA + "tldnorthwest/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestldnorthwest.xlsx",
    "uk_gva_yorkshire.xlsx":        ONS + GVA + "tleyorkshireandthehumber/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestleyorkshireandthehumber.xlsx",
    "uk_gva_south_west.xlsx":       ONS + GVA + "tlksouthwest/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestlksouthwest.xlsx",
    "uk_gva_wales.xlsx":            ONS + GVA + "tllwales/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestllwales.xlsx",
    "uk_gva_scotland.xlsx":         ONS + GVA + "tlmscotland/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestlmscotland.xlsx",
    "uk_gva_northern_ireland.xlsx": ONS + GVA + "tlnnorthernireland/regionalgrossvalueaddedbalancedbyindustrylocalauthoritiestlnnorthernireland.xlsx",
    # --- Markets: online share of retail ------------------------------------------
    "uk_internet_share_of_retail_J4MC.csv": "https://www.ons.gov.uk/generator?format=csv&uri=/businessindustryandtrade/retailindustry/timeseries/j4mc/drsi",
    "us_ecommerce_share_of_retail_ECOMPCTSA.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=ECOMPCTSA",
    # --- Markets: total retail sales and exchange rates (market size and growth) ---
    "uk_retail_weekly_value_JE4W.csv": "https://www.ons.gov.uk/generator?format=csv&uri=/businessindustryandtrade/retailindustry/timeseries/je4w/drsi",
    "us_retail_trade_MRTSSM44000USN.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MRTSSM44000USN",
    "fx_usd_per_gbp_AEXUSUK.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=AEXUSUK",
    "fx_twd_per_usd_AEXTAUS.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=AEXTAUS",
}
HEADERS = {"User-Agent": "Mozilla/5.0 (portfolio project; contact via GitHub)"}

In [2]:
log = []
for name, url in SOURCES.items():
    path = os.path.join(RAW, name)
    try:
        r = requests.get(url, headers=HEADERS, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        log.append({"file": name, "status": "OK", "kb": round(len(r.content) / 1024)})
    except Exception as e:
        log.append({"file": name, "status": f"FAILED: {e}"[:90], "kb": 0})

log = pd.DataFrame(log)
log["downloaded_on"] = datetime.date.today().isoformat()
log.to_csv(os.path.join(RAW, "_download_log.csv"), index=False)
log

,file,status,kb,downloaded_on
0,uk_population_mid2025.xlsx,OK,1003,2026-10-22
1,uk_gdhi_local_authorities_1997_2024.xlsx,OK,6101,2026-10-22
2,uk_ashe_table8_2025_provisional.zip,OK,8199,2026-10-22
3,uk_gva_london.xlsx,OK,1203,2026-10-22
4,uk_gva_west_midlands.xlsx,OK,1075,2026-10-22
5,uk_gva_north_west.xlsx,OK,1252,2026-10-22
6,uk_gva_yorkshire.xlsx,OK,564,2026-10-22
7,uk_gva_south_west.xlsx,OK,760,2026-10-22
8,uk_gva_wales.xlsx,OK,515,2026-10-22
9,uk_gva_scotland.xlsx,OK,971,2026-10-22


## Manual inputs (figures published only in reports)

Dublin is in the Republic of Ireland, so its figures come from the Central Statistics Office (CSO), not the ONS.

In [4]:
manual = pd.DataFrame([
    {"city": "Dublin", "measure": "population", "value": 1_597_700, "unit": "people", "period": "April 2026",
     "source": "CSO, Population and Migration Estimates April 2026 (Dublin region)",
     "url": "https://www-cloud.cso.ie/en/releasesandpublications/ep/p-pme/populationandmigrationestimatesapril2026/keyfindings/"},
    {"city": "Dublin", "measure": "disposable_income_per_person", "value": 32_393, "unit": "EUR", "period": "2023",
     "source": "CSO, County Incomes and Regional GDP 2022-2023",
     "url": "https://www.cso.ie/en/releasesandpublications/ep/p-cirgdp/countyincomesandregionalgdp2022-2023/disposableincomebycounty"},
])
manual.to_csv(os.path.join(RAW, "ireland_manual_inputs.csv"), index=False)
manual

,city,measure,value,unit,period,source,url
0,Dublin,population,1597700,people,April 2026,"CSO, Population and Migration Estimates April ...",https://www-cloud.cso.ie/en/releasesandpublica...
1,Dublin,disposable_income_per_person,32393,EUR,2023,"CSO, County Incomes and Regional GDP 2022-2023",https://www.cso.ie/en/releasesandpublications/...
